# Ideal SQR Feasibility with Direct and Echoed Multitone Waveforms

This notebook reproduces the main saved results for the follow-up study comparing direct and echoed multitone parameterizations for an ideal SQR gate. The default path is fast: it loads saved artifacts, figures, and validation outputs. Commented rerun cells are provided for users who want to regenerate a selected case or the full study from the same parameter block.

## Environment Setup

This cell imports the study helpers, finds the study root, and exposes the shared utilities used later in the notebook. The tunable parameters below affect which saved cases are inspected and, if you enable reruns, which model and target are rebuilt.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

study_dir = Path.cwd()
if study_dir.name != 'ideal_sqr_direct_vs_echoed_multitone':
    if (study_dir / 'studies' / 'ideal_sqr_direct_vs_echoed_multitone').exists():
        study_dir = study_dir / 'studies' / 'ideal_sqr_direct_vs_echoed_multitone'
    else:
        for parent in [study_dir, *study_dir.parents]:
            candidate = parent / 'studies' / 'ideal_sqr_direct_vs_echoed_multitone'
            if candidate.exists():
                study_dir = candidate
                break
scripts_dir = study_dir / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from common import build_frame, build_model, build_target_operator, duration_from_chi_t, logical_levels, target_spec

data_dir = study_dir / 'data'
figures_dir = study_dir / 'figures'
artifacts_dir = study_dir / 'artifacts'
print(f'Study directory: {study_dir}')
print(f'Data directory:  {data_dir}')
print(f'Figures dir:     {figures_dir}')

## User-Tunable Parameters

All user-adjustable knobs live in this single cell. Change them here and rerun the next derived-objects cell to update the downstream state. The defaults are chosen to inspect the best direct case and the best echoed case saved by the study.

In [ ]:
# --- User-tunable parameters ---
model_variant = 'chi_only'
include_chi_prime = False
target_family = 'smooth_x'
n_active = 2
chi_t_over_2pi = 5.0
construction_focus = 'direct_multitone'

echo_model_variant = 'chi_plus_chiprime'
echo_include_chi_prime = True
echo_target_family = 'staggered_x'
echo_n_active = 2
echo_chi_t_over_2pi = 3.0
echo_construction_focus = 'echoed_asymmetric'

rerun_dt_s = 4.0e-9
rerun_n_cav_padding = 2
rerun_full_study = False
rerun_validation = False

active_config = {
    'model_variant': model_variant,
    'include_chi_prime': include_chi_prime,
    'target_family': target_family,
    'n_active': n_active,
    'chi_t_over_2pi': chi_t_over_2pi,
    'construction_focus': construction_focus,
    'echo_model_variant': echo_model_variant,
    'echo_include_chi_prime': echo_include_chi_prime,
    'echo_target_family': echo_target_family,
    'echo_n_active': echo_n_active,
    'echo_chi_t_over_2pi': echo_chi_t_over_2pi,
    'echo_construction_focus': echo_construction_focus,
    'rerun_dt_s': rerun_dt_s,
    'rerun_n_cav_padding': rerun_n_cav_padding,
}
print(json.dumps(active_config, indent=2))

## Derived Objects

This cell rebuilds the study objects implied by the parameter block: the dispersive model, frame, logical subspace, target operator, case identifiers, and default artifact paths. If you change any tunable parameter above, rerun this cell before loading artifacts or attempting a live rerun.

In [ ]:
case_id = f"{model_variant}_{target_family}_na{int(n_active)}_chiT{str(chi_t_over_2pi).replace('.', 'p')}"
echo_case_id = f"{echo_model_variant}_{echo_target_family}_na{int(echo_n_active)}_chiT{str(echo_chi_t_over_2pi).replace('.', 'p')}"

model = build_model(include_chi_prime=include_chi_prime, n_active=n_active, n_cav_padding=rerun_n_cav_padding)
frame = build_frame(model)
levels = logical_levels(n_active)
spec = target_spec(target_family, n_active)
target_operator = build_target_operator(spec, levels)
duration_s = duration_from_chi_t(chi_t_over_2pi)

echo_model = build_model(include_chi_prime=echo_include_chi_prime, n_active=echo_n_active, n_cav_padding=rerun_n_cav_padding)
echo_frame = build_frame(echo_model)
echo_levels = logical_levels(echo_n_active)
echo_spec = target_spec(echo_target_family, echo_n_active)
echo_target_operator = build_target_operator(echo_spec, echo_levels)
echo_duration_s = duration_from_chi_t(echo_chi_t_over_2pi)

direct_artifact_path = artifacts_dir / 'cases' / f'{case_id}_{construction_focus}.json'
echo_artifact_path = artifacts_dir / 'cases' / f'{echo_case_id}_{echo_construction_focus}.json'

print(f'Direct case: {case_id} ({construction_focus})')
print(f'Echo case:   {echo_case_id} ({echo_construction_focus})')
print(f'Direct duration: {1e9 * duration_s:.3f} ns')
print(f'Echo duration:   {1e9 * echo_duration_s:.3f} ns')
print(f'Direct artifact exists: {direct_artifact_path.exists()}')
print(f'Echo artifact exists:   {echo_artifact_path.exists()}')

## Step 1: Load Saved Study Summary

This cell loads the saved summary tables and reproduces the main high-level comparison between constructions. These outputs depend on the saved study data, not on a live rerun, so this is the fastest way to inspect the main result.

In [ ]:
# --- Load saved results (default) ---
summary = json.loads((data_dir / 'study_summary.json').read_text(encoding='utf-8-sig'))
construction_df = pd.DataFrame(summary['construction_summary'])
target_df = pd.DataFrame(summary['target_construction_summary'])
display(Markdown('### Construction-level summary'))
display(construction_df)
display(Markdown('### Target-family summary'))
display(target_df)
display(Markdown(f"Best overall saved row: **{summary['best_overall']['construction']}** on **{summary['best_overall']['case_id']}** with average fidelity **{summary['best_overall']['average_gate_fidelity']:.6f}**."))

This optional rerun cell recomputes the full study outputs from the scripts using the current environment. It is expensive and is commented out by default. The tunable parameters above do not change the full grid; they only help you inspect chosen saved cases unless you modify the study script itself.

In [ ]:
# --- Re-run with current parameters ---
# from subprocess import run
# if rerun_full_study:
#     run([sys.executable, str(scripts_dir / 'run_study.py')], check=True)

## Step 2: Load Saved Best-Case Artifacts

This cell loads one direct artifact and one echoed artifact chosen from the parameter block above, then extracts the main metrics and a few key optimized parameters. These outputs depend on the case identifiers, target family, and construction names in the parameter cell.

In [ ]:
# --- Load saved results (default) ---
direct_artifact = json.loads(direct_artifact_path.read_text(encoding='utf-8-sig'))
echo_artifact = json.loads(echo_artifact_path.read_text(encoding='utf-8-sig'))

direct_row = direct_artifact['summary_row']
echo_row = echo_artifact['summary_row']

compare_df = pd.DataFrame([
    {
        'label': 'direct',
        'case_id': direct_row['case_id'],
        'construction': direct_row['construction'],
        'average_gate_fidelity': direct_row['average_gate_fidelity'],
        'mean_residual_z_error_rad': direct_row['mean_residual_z_error_rad'],
        'mean_transverse_error_rad': direct_row['mean_transverse_error_rad'],
        'total_gate_duration_ns': direct_row['total_gate_duration_ns'],
    },
    {
        'label': 'echo',
        'case_id': echo_row['case_id'],
        'construction': echo_row['construction'],
        'average_gate_fidelity': echo_row['average_gate_fidelity'],
        'mean_residual_z_error_rad': echo_row['mean_residual_z_error_rad'],
        'mean_transverse_error_rad': echo_row['mean_transverse_error_rad'],
        'total_gate_duration_ns': echo_row['total_gate_duration_ns'],
    },
])
display(compare_df)

display(Markdown('### Direct optimized corrections'))
display(pd.DataFrame(direct_artifact['metadata']['corrections_segment_1']))
display(Markdown('### Echo segment 1 optimized corrections'))
display(pd.DataFrame(echo_artifact['metadata']['corrections_segment_1']))
display(Markdown('### Echo segment 2 optimized corrections'))
display(pd.DataFrame(echo_artifact['metadata']['corrections_segment_2']))

This optional rerun cell recomputes a single selected case with the saved study script. Update the parameter block above first, then uncomment this cell if you want a live rerun. This is much cheaper than rerunning the entire study grid.

In [ ]:
# --- Re-run with current parameters ---
# from subprocess import run
# run([sys.executable, str(scripts_dir / 'run_study.py'), '--case', case_id], check=True)
# run([sys.executable, str(scripts_dir / 'run_study.py'), '--case', echo_case_id], check=True)

## Step 3: Validation Outputs

This cell loads the saved sanity and convergence checks. These outputs depend on the saved study summary and artifacts and provide the main numerical validation for the report.

In [ ]:
# --- Load saved results (default) ---
validation = json.loads((data_dir / 'validation_summary.json').read_text(encoding='utf-8-sig'))
display(pd.DataFrame(validation['convergence_rows']))
display(Markdown(f"Sanity check pass: **{validation['sanity_case']['pass']}**; direct fidelity = **{validation['sanity_case']['direct_fidelity']:.6f}**, symmetric echo fidelity = **{validation['sanity_case']['symmetric_echo_fidelity']:.6f}**."))
display(Markdown(f"Convergence pass: **{validation['convergence_pass']}**."))

This optional rerun cell rebuilds the validation summary from the saved artifacts. Use it after regenerating study data if you want to refresh the convergence and sanity checks.

In [ ]:
# --- Re-run with current parameters ---
# from subprocess import run
# if rerun_validation:
#     run([sys.executable, str(scripts_dir / 'validate_results.py')], check=True)

## Step 4: Key Figures

This cell redisplays the saved publication figures used in the report. The images are generated by the study script and can be compared visually against a fresh rerun if desired.

In [ ]:
for figure_name in [
    'construction_metric_means.png',
    'duration_fidelity_tradeoff.png',
    'echo_delta_tradeoff.png',
    'case_construction_heatmap.png',
    'representative_waveforms.png',
]:
    display(Markdown(f'### {figure_name}'))
    display(Image(filename=str(figures_dir / figure_name)))

## Summary

This notebook reproduced the saved comparison between direct and echoed multitone ideal-SQR parameterizations, the best-case artifacts, the validation checks, and the report figures. The saved data show that the direct multitone waveform is the strongest candidate in this study, while the echoed constructions do not deliver practical residual-$Z$ cancellation once finite refocusing pulses are included.

| Tunable parameter | Default value | Effect on notebook outputs |
| --- | --- | --- |
| `model_variant` / `include_chi_prime` | `chi_only` / `False` | Chooses the direct saved case and rebuilds the direct dispersive model. |
| `target_family` | `smooth_x` | Chooses which ideal-SQR angle family is inspected for the direct case. |
| `n_active` | `2` | Sets the direct logical subspace size and target operator dimension. |
| `chi_t_over_2pi` | `5.0` | Sets the direct active SQR duration and direct case identifier. |
| `construction_focus` | `direct_multitone` | Chooses which saved direct-style artifact is loaded. |
| `echo_model_variant` / `echo_include_chi_prime` | `chi_plus_chiprime` / `True` | Chooses the echoed saved case and rebuilds the echoed dispersive model. |
| `echo_target_family` | `staggered_x` | Chooses which echoed target family is inspected. |
| `echo_n_active` | `2` | Sets the echoed logical subspace size and target operator dimension. |
| `echo_chi_t_over_2pi` | `3.0` | Sets the echoed active SQR duration and echoed case identifier. |
| `echo_construction_focus` | `echoed_asymmetric` | Chooses which saved echoed artifact is loaded. |
| `rerun_dt_s` / `rerun_n_cav_padding` | `4e-9` / `2` | Control the rebuild settings used by any commented live-rerun workflow you enable manually. |
| `rerun_full_study` / `rerun_validation` | `False` / `False` | Guard the optional expensive rerun cells. |

Caveat: the notebook defaults to loading saved data because the full study takes several minutes. Uncomment the rerun cells only when you actually want live recomputation.